---
# FNR Self-Adaptation for a Weakly Coupled MDP
---

This notebook builds the fashion-assortment WMDP, solves the Lagrangian baseline, then solves the feasible network relaxation (FNR). The final cells compare the FNR policy against the baseline policy.


## Modules and Initial Setup

To set up required packages when using Conda, use:

`conda env create -f environment.yml`


In [ ]:
import inspect
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import clear_output, display


def find_project_root(start_path: Path) -> Path:
    """Find the weakly-coupled MDP project root from a notebook directory."""
    for candidate in (start_path, *start_path.parents):
        if (candidate / "wmdp.py").exists() and (candidate / "fnr.py").exists():
            return candidate
    raise RuntimeError("Could not locate the weakly-coupled MDP project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from wmdp import *
from fnr import *
from delayedallocation import *
from lagrangian import *
from simulator import Simulator

print("Project root:", PROJECT_ROOT)


## WMDP Example

We use a small <b>dynamic assortment</b> example on fashion retail assortment planning. It is a stylized tutorial instance motivated by fashion demand and merchandising problems. It is inspired by <i>Caro F, Gallien J (2007) -  Dynamic assortment with demand learning for seasonal consumer goods. Management Science 53(2):276–292.</i>

A retail merchandising team follows five items over five weekly decision periods. Each week, the team decides which items should receive featured placement, subject to shared limits on shelf-display space and visual-merchandising budget.

<b>States</b>

Each item starts in the `high_demand` state in period 0. From period 1 onward, each item can be in one of two states:

- `high_demand`: the item is in a high-demand condition this week.
- `low_demand`: the item is in a low-demand condition and may need promotional support.

The fixed all-high-demand initial state can still reach every later joint state: under standard listing, each high_demand item independently remains high demand with probability `0.8` or shifts to low demand with probability `0.2`.

The reward is a weekly merchandise value score. A positive score means a successful assortment week: a high_demand item is featured while demand remains strong. The assortment value differs by item, with scores `[1, 1, 2, 4, 8]`. A score of `0.0` means a neutral merchandising week. A score of `-1.0` means unfeatured low demand: a low_demand item is left in the standard catalog.

<b>Actions</b>

For each item and week, the decision maker chooses one of two actions:

- `0`: standard listing only. The item remains available in the standard catalog, but it is not selected for the featured assortment.
- `1`: featured placement. The item is selected for the featured assortment, using shelf-display space and visual-merchandising budget.

<b>Linking Constraints</b>

The action is chosen separately for each item, but the actions are coupled by shared weekly resource limits. Featured placement consumes two shared resources: shelf-display space and visual-merchandising budget. Different items require different amounts of these resources, so the feature coefficients vary across items.


## Constructing the WMDP

The next cells translate the fashion-assortment story into the data structures used by the WMDP code: binary action sets, one component MDP per item, and two linking constraints for the shared weekly resources.


### Action Set

The next code cell creates the same binary action set for all five items. In the example, `0` means standard listing and `1` means featured placement.


In [ ]:
# J is the number of item components and T is the number of weekly periods.
# Each item has the same binary action set: 0 = standard listing, 1 = featured placement.
J = 5
T = 5
action_sets = [[0, 1] for _ in range(J)]

print("Action sets:")
action_sets

### Component MDPs

The next code cell builds the item-level MDPs. All items start `high_demand` in period 0, and each later period has both possible item states. The transition model is shared across items, while the featured-assortment reward varies by item.


In [ ]:
# Item-specific assortment value scores for featured placement in the high_demand state.
# Later items have higher assortment value, which makes assortment allocation more consequential.
high_demand_feature_reward = [1.0, 1.0, 2.0, 4.0, 8.0]

# Each transition key is (current_state, action, next_state), and the value is the transition probability.
# For example, the first line means that a high_demand item that is left in the standard catalog
# remains high demand next week with probability 0.8.
transition_kernel = {
    ("high_demand", 0, "high_demand"): 0.8,
    ("high_demand", 0, "low_demand"): 0.2,
    ("high_demand", 1, "high_demand"): 1.0,
    ("low_demand", 0, "low_demand"): 1.0,
    ("low_demand", 1, "high_demand"): 0.7,
    ("low_demand", 1, "low_demand"): 0.3,
}

# Build one component MDP for each item.
# Period 0 contains only the fixed initial state; later periods contain all reachable states.
components = []
for j in range(J):
    state_reward_by_period = [
        [("high_demand", {0: 0.0, 1: high_demand_feature_reward[j]})],
    ] + [
        [
            ("high_demand", {0: 0.0, 1: high_demand_feature_reward[j]}),
            ("low_demand", {0: -1.0, 1: 0.0}),
        ]
        for _ in range(T - 1)
    ]

    component = build_component(
        component=j,
        actions=action_sets[j],
        state_data_by_period=state_reward_by_period,
        transitions_by_period=[transition_kernel.copy() for _ in range(len(state_reward_by_period) - 1)],
    )
    components.append(component)

# Example: printing the first item component
print("Component 0, period-0 states:")
print(components[0].states[0])
print()
print("Component 0 transition kernel for period 0:")
print(components[0].P[0])

### Linking Constraints

The next code cell adds the weekly resource limits. The first coefficient dictionary is shelf-display space and the second is visual-merchandising budget. Action `0` has omitted coefficients, so it consumes zero resources; action `1` consumes item-specific amounts.

The shelf-display-space budget is

$$
3 \times \mathbf{1}\{a_0 = 1\} + 3 \times \mathbf{1}\{a_1 = 1\} + 8 \times \mathbf{1}\{a_2 = 1\} + 8 \times \mathbf{1}\{a_3 = 1\} + 8 \times \mathbf{1}\{a_4 = 1\} \leq 14.
$$

The visual-merchandising budget is

$$
5 \times \mathbf{1}\{a_0 = 1\} + 5 \times \mathbf{1}\{a_1 = 1\} + 13 \times \mathbf{1}\{a_2 = 1\} + 13 \times \mathbf{1}\{a_3 = 1\} + 13 \times \mathbf{1}\{a_4 = 1\} \leq 23.
$$


In [ ]:
# Each resource dictionary is keyed by (item_index, action).
# Only action 1 appears because standard listing consumes zero featured placement resources by default.

# For example, shelf_display_space[(0, 1)] = 3.0 means featured placement for item 0 uses 3 units of shelf-display space.
# Similarly, visual_merchandising[(0, 1)] = 5.0 means featured placement for item 0 uses 5 units of visual-merchandising budget.
shelf_display_space = {
    (0, 1): 3.0,
    (1, 1): 3.0,
    (2, 1): 8.0,
    (3, 1): 8.0,
    (4, 1): 8.0,
}
visual_merchandising = {
    (0, 1): 5.0,
    (1, 1): 5.0,
    (2, 1): 13.0,
    (3, 1): 13.0,
    (4, 1): 13.0,
}

# The two right-hand sides are the weekly shelf-display-space and visual-merchandising budgets.
linking_constraints = build_linking_constraints(
    action_sets=action_sets,
    constraint_coefficients=[shelf_display_space, visual_merchandising],
    rhs_values=[14.0, 23.0],
)

print("Linking constraints:")
print(linking_constraints)

### Assembling the WMDP

The next code cell combines the five item MDPs with the shared resource constraints. The period-0 joint state is now fixed to all items in `high_demand`; later periods contain the full reachable state space.


In [ ]:
# Combine the item component MDPs and the shared resource constraints into one WMDP.
wmdp = build_wmdp(
    components=components,
    linking_constraints=linking_constraints,
)

print(wmdp)


### The WMDP size

If enumerating states and actions for a classical dynamic programming forward/backward recursion, or to construct an exact linear program, what would be the size of the state and action spaces?

In [ ]:
# Count the joint states and joint actions that an exact dynamic program or exact LP would enumerate.
state_counts_by_period = {
    t: len(wmdp.generate_states(t))
    for t in range(wmdp.T)
}

# The feasible joint action space keeps only featured placement plans satisfying the two resource budgets.
feasible_actions = [
    tuple(action)
    for action, _ in wmdp.generate_feasible_actions()
]
feasible_action_count = len(feasible_actions)

# An exact recursion or exact LP would consider every feasible action in every joint state.
state_action_counts_by_period = {
    t: state_counts_by_period[t] * feasible_action_count
    for t in range(0, wmdp.T)
}

total_joint_states = sum(state_counts_by_period.values())
total_state_action_pairs = sum(state_action_counts_by_period.values())

print("Exact state-action enumeration using feasible actions:")
for t, count in state_action_counts_by_period.items():
    print(f"  period {t}: {state_counts_by_period[t]} states x {feasible_action_count} actions = {count}")
print(f"Total state-action pairs across {wmdp.T} periods: {total_state_action_pairs}")
print()

## The baseline: Lagrangian relaxation

As a baseline, we solve a relaxation that enforces the shelf-display-space and visual-merchandising constraints only in expectation. The resulting marginal flows are easy to compute, but the sampled policy still has to be repaired online to return a feasible weekly featured placement plan.


### <b>Construct</b>: Building the Lagrangian Relaxation

The next code cell constructs the expectation-relaxed model for the same fashion-assortment WMDP. It uses item-level marginal flow variables and replaces pathwise feasibility with expected resource-use constraints in each period.


In [ ]:
# Construct the expectation-relaxed Lagrangian model.
# The model keeps component-level marginal flows and enforces resources only in expectation.
lagrangian_model = Lagrangian(wmdp)


### <b>Optimize</b>: Solving the Lagrangian Relaxation

The next code cell solves the relaxed LP and extracts a sampling policy. The reported objective is the relaxation value; the policy simulation below measures the feasible repaired policy that is actually executed.


In [ ]:
# Solve the Lagrangian relaxation and inspect expected resource use by period and constraint.
lagrangian_result = lagrangian_model.optimize()

print("Objective value: ", lagrangian_result.objective_value)

print("\nExpected resource use:")
for (period, constraint), value in lagrangian_result.expected_resource_use.items():
    budget = wmdp.linking_constraints.b[constraint]
    print(f"  period {period}, constraint {constraint}: {value:.3f} / {budget:.3f}")


### Simulating the Lagrangian Policy

The next code cell simulates the feasible policy induced by the Lagrangian marginals. At each state, the policy samples item actions one at a time from the marginal probabilities; if a sampled featured placement action would violate the original resource constraints, it uses standard listing action `0` for that item.


In [ ]:
# Simulate the repaired Lagrangian policy and plot total realized outcome scores.
num_simulations = 100
lagrangian_simulator = Simulator(wmdp, lagrangian_result.policy)
lagrangian_total_rewards = []

fig, ax = plt.subplots(figsize=(8, 4))

for simulation_index in range(num_simulations):
    simulation_result = lagrangian_simulator.simulate()
    lagrangian_total_rewards.append(simulation_result["total_reward"])

    ax.clear()
    ax.hist(
        lagrangian_total_rewards,
        bins=min(30, max(1, len(set(lagrangian_total_rewards)))),
        color="steelblue",
        edgecolor="white",
    )
    sample_mean = sum(lagrangian_total_rewards) / len(lagrangian_total_rewards)
    ax.axvline(
        sample_mean,
        color="darkorange",
        linewidth=2,
        label=f"mean: {sample_mean:.2f}",
    )
    ax.axvline(
        lagrangian_result.objective_value,
        color="seagreen",
        linewidth=2,
        linestyle="--",
        label=f"relaxation: {lagrangian_result.objective_value:.2f}",
    )
    ax.set_title(f"Lagrangian policy simulation ({simulation_index + 1}/{num_simulations})")
    ax.set_xlabel("Total reward")
    ax.set_ylabel("Frequency")
    ax.legend(loc="upper right")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

# clear_output(wait=True)
# display(fig)
print(f"Mean total reward over {num_simulations} simulations: {sum(lagrangian_total_rewards) / len(lagrangian_total_rewards):.3f}")


## Self-Adapting Network Relaxations

We now build and solve the feasible network relaxation (FNR) for the fashion-assortment WMDP. The network represents the feasible combinations of featured placement decisions under the two resource budgets.


### <b>Construct</b>: Building the Feasibility Network Relaxation

The next code cell constructs the reduced layered network from the linking constraints. Each path through the network corresponds to a feasible weekly featured placement plan across the five items.


In [ ]:
# Build the FNR network from the two resource constraints.
# Network paths encode feasible joint featured placement decisions for one week.
network = construct_fnr_network(wmdp.linking_constraints)
print("Network size:")
print(network.get_size())

### Visualizing the Network

The next code cell draws the feasibility network. Layers correspond to items, and arcs correspond to choosing standard listing or featured placement while accumulating resource use.


In [ ]:
# Draw the layered feasibility network: item layers, action arcs, and accumulated resource use.
draw_fnr_network(network, figsize=(10, 3))

<b>How many actions does this represent?</b>

The next code enumerates all feasible actions, each represented as a path of the network.

In [ ]:
feasible_actions = [
    tuple(action)
    for action, _ in wmdp.generate_feasible_actions()
]

print(f"Feasible actions (total: {len(feasible_actions)}):")
for action in feasible_actions:
    print("\tAction " + str(action))

<b>Linear programming construction</b>

In [ ]:
# Construct the FNR linear-programming model for the fashion-assortment WMDP.
FNR_model = FNR(wmdp, network)

## <b>Optimize</b>: Solving the FNR Model

The next code cell solves the FNR linear program for the featured placement problem. The objective is the expected total weekly merchandise value score over the horizon, and the solution also gives marginal action flows and an induced policy.


In [ ]:
# Solve the FNR model and report the expected total outcome score and marginal action flows.
fnr_result = FNR_model.optimize()

print("Objective value: ", fnr_result.objective_value)
print("Marginals:", fnr_result.marginal_flows)

### Simulating the FNR Policy

The next code cell queries the policy extracted from the FNR network flows and then simulates it. The policy follows positive network flow when available.


In [ ]:
# Simulate the FNR policy and plot total realized outcome scores.
num_simulations = 100
fnr_simulator = Simulator(wmdp, fnr_result.policy)
fnr_total_rewards = []

fig, ax = plt.subplots(figsize=(8, 4))

for simulation_index in range(num_simulations):
    simulation_result = fnr_simulator.simulate()
    fnr_total_rewards.append(simulation_result["total_reward"])

    ax.clear()
    ax.hist(
        fnr_total_rewards,
        bins=min(30, max(1, len(set(fnr_total_rewards)))),
        color="steelblue",
        edgecolor="white",
    )
    sample_mean = sum(fnr_total_rewards) / len(fnr_total_rewards)
    ax.axvline(
        sample_mean,
        color="darkorange",
        linewidth=2,
        label=f"mean: {sample_mean:.2f}",
    )
    ax.axvline(
        fnr_result.objective_value,
        color="seagreen",
        linewidth=2,
        linestyle="--",
        label=f"model: {fnr_result.objective_value:.2f}",
    )
    ax.set_title(f"FNR policy simulation ({simulation_index + 1}/{num_simulations})")
    ax.set_xlabel("Total reward")
    ax.set_ylabel("Frequency")
    ax.legend(loc="upper right")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

clear_output(wait=True)
display(fig)
print(f"Mean total reward over {num_simulations} simulations: {sum(fnr_total_rewards) / len(fnr_total_rewards):.3f}")


## Value of Self-Adaptation Relative to the Baseline

The Lagrangian objective is a relaxation value, while the simulations estimate the value of the feasible policy that is actually executed. The comparison below focuses on the simulated policy means and also reports the model objectives for context.


In [ ]:
lagrangian_mean = sum(lagrangian_total_rewards) / len(lagrangian_total_rewards)
fnr_mean = sum(fnr_total_rewards) / len(fnr_total_rewards)

print("Model objective values:")
print(f"  Lagrangian relaxation: {lagrangian_result.objective_value:.3f}")
print(f"  FNR model:             {fnr_result.objective_value:.3f}")
print()
print("Simulated feasible policy values:")
print(f"  Lagrangian policy mean: {lagrangian_mean:.3f}")
print(f"  FNR policy mean:        {fnr_mean:.3f}")
print(f"  FNR improvement:        {fnr_mean - lagrangian_mean:.3f}")
